In [1]:
from sympy import (
    Function, Symbol, symbols, sqrt, diff, simplify, Eq, solve, dsolve, latex, Integral
)
from sympy import Equality, Limit
from sympy import *
from IPython.display import display, Math

In [2]:
# Derive Euler-Lagrange via one-parameter variation η

x = Symbol("x", real=True)
y = Function("y")(x)
F = Function("F")(x, y, y.diff(x))
objective = Integral(F, (x))

eps = Symbol("varepsilon", real=True)
eta = Function("eta")(x)
y_eps = y + eps * eta

# shouldn't use {y: y_eps, y.diff(x): y_eps.diff(x)}, becasue sympy replace y_eps.diff(x) at first, 
# then it will add new y(x), then this y(x) will be replaced by y_eps, then cause the third term be replaced two times
F_eps = F.subs({y: y_eps})
# display(Math(latex(F_eps)))

dF = Integral(F_eps - F, (x))  # finite change of F when adding a small arbitrary perturbation eta
dF_eta = dF / eps  # difference quotient like definition of derivative
dJ = Limit(dF_eta, eps, 0)  # functional gradient
# display(Math(latex(dJ)))

equ = Equality(dJ, 0)  # when dJ = 0, means we arrive extreme value

# now let's consider F_eps - F
# taylor expansion of F_eps
F_lin = F_eps.series(eps, 0, 2).removeO()
delta_J = Integral((F_lin - F) / eps, (x))
display(Math(latex(delta_J)))

integrand = (F_lin - F) / eps

def ibp_on_eta(integrand, eta, x, a, b):
    """∫ U η' dx = [U η]_a^b − ∫ (dU/dx) η dx"""
    d_eta = diff(eta, x)
    U = Wild("U", exclude=[d_eta])

    unchange = 0   # terms without η'
    U_sum = 0      # sum of factors multiplying η'
    for term in Add.make_args(expand(integrand)):
        m = term.match(U * d_eta)
        if m is None:
            unchange += term
        else:
            U_sum += m[U]

    # full analytical expression (volume + boundary)
    boundary = (U_sum * eta).subs(x, b) - (U_sum * eta).subs(x, a)    
    return (
        Integral(unchange, (x, a, b))
        - Integral(Derivative(U_sum, x) * eta, (x, a, b))
        + boundary
    )

a, b = symbols("a b", real=True)
after = ibp_on_eta(integrand, eta, x, a, b)
# display(Math(latex(after)))

# fixed endpoints η(a)=η(b)=0 → drop boundary, factor η → EL
delta_J_fixed = after.subs({eta.subs(x, a): 0, eta.subs(x, b): 0})
# display(Math(latex(delta_J_fixed)))

integrand_sum = 0
for term in Add.make_args(delta_J_fixed):
    coeff, integ = term.as_coeff_Mul()
    integrand_sum += coeff * integ.function  # type: ignore
combined = Integral(collect(integrand_sum, eta), (x, a, b))

# Fundamental lemma: ∫ E(x) η(x) dx = 0 for all η (η(a)=η(b)=0)
# ⇒ E(x) = 0 everywhere
E = combined.function.coeff(eta)   # type: ignore : Euler–Lagrange expression
el_eq = Eq(E, 0)

display(Math(latex(combined)))
display(Math(latex(el_eq)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# Concrete shortest-path Lagrangian: F = √(1 + (y')²)
x = Symbol("x", real=True)
y = Function("y")(x)

dx = symbols('dx', positive=True)
dy = y.diff(x) * dx
dL = simplify(sqrt(dx**2 + dy**2))
dL_dx = simplify(dL / dx)   # = sqrt(1 + y'(x)**2)

a, b = symbols("a b", real=True)
L = Integral(dL_dx, (x, a, b))
display(Math(latex(L)))

# Substitute this integrand for abstract F in the EL equation from above.
# F(x, y, y') → √(1 + (y')²)  (no explicit x, y dependence)
# Note: substitute dL_dx's form, NOT the Integral L
el_arc = el_eq.replace(Function("F"), lambda *args: sqrt(1 + args[2]**2))
el_arc = simplify(el_arc.doit())   # evaluate Subs(...), then simplify

display(Math(latex(el_arc)))
# ⇒ y''(x) / (1 + y'(x)²)^{3/2} = 0  ⇒  y'' = 0  (straight line)

# Analytic solution of the EL ODE
sol = dsolve(el_arc, y)                      # y(x) = C1 + C2*x
display(Math(latex(sol)))

# With fixed endpoints y(a)=ya, y(b)=yb
ya, yb = symbols("y_a y_b", real=True)
sol_bc = dsolve(el_arc, y, ics={y.subs(x, a): ya, y.subs(x, b): yb})
sol_bc = simplify(sol_bc)  # type: ignore
display(Math(latex(sol_bc)))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>